# 의사역행렬과 추천 시스템

> 선형대수 18강 · 특이값 분해

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [의사역행렬과 추천 시스템](https://mioon1402.github.io/timeseriesdata/linalg/L18-pseudoinverse.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 역행렬이 없다는 것

## 1. 해가 무한히 많을 때 — 최소노름해

## 2. SVD 로 A⁺ 를 만든다

## 3. 해가 없을 때 — 최소제곱해

## 4. A⁺ 의 정의 — 무어-펜로즈 네 조건

## 5. 추천 시스템 — 빈칸 채우기

## 6. 극분해 — 회전 × 늘림

## 7. numpy 로 확인하기

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[1., 2.]])          # 식 1개, 미지수 2개
b = np.array([3.])

x_plus = np.linalg.pinv(A) @ b
x_alt  = np.array([3., 0.])       # 눈으로 찾은 다른 해

print("A⁺b   =", x_plus, " 길이 =", round(float(np.linalg.norm(x_plus)), 4))
print("다른 해 =", x_alt,   " 길이 =", round(float(np.linalg.norm(x_alt)), 4))
print()
print("둘 다 식을 만족하는가:", A @ x_plus, A @ x_alt)
print("A⁺b 는 행공간 (1,2) 의 배수인가:", x_plus / np.array([1., 2.]))

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[3., 1., 4.],
              [1., 0., 1.],
              [4., 1., 5.]])      # 3열 = 1열 + 2열 → 랭크 2 (특이!)

U, s, Vt = np.linalg.svd(A)
print("특이값 =", s)               # 마지막이 사실상 0
print("랭크  =", np.linalg.matrix_rank(A))

# Σ⁺ : 0 이 아닌 σ 만 뒤집는다
tol = s.max() * 1e-10
s_plus = np.array([1/x if x > tol else 0.0 for x in s])
A_plus = Vt.T @ np.diag(s_plus) @ U.T

print("\n수동 A⁺ 와 np.linalg.pinv 가 같은가:",
      np.allclose(A_plus, np.linalg.pinv(A)))
print("A⁻¹ 는? →", end=" ")
try:
    np.linalg.inv(A)
except np.linalg.LinAlgError as e:
    print("불가능:", e)

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[1., 0.],
              [1., 1.],
              [1., 2.],
              [1., 3.]])          # 4×2, 세로로 긴 행렬
Ap = np.linalg.pinv(A)

print("A⁺A (2×2) =\n", Ap @ A)     # ← 이건 I 가 맞다 (열이 독립)
print("\nAA⁺ (4×4) =\n", A @ Ap)   # ← 이건 I 가 아니다
print("\nAA⁺ 는 대칭인가:", np.allclose((A@Ap).T, A@Ap))
print("AA⁺ 를 제곱하면 자기 자신인가:", np.allclose((A@Ap) @ (A@Ap), A@Ap))
print("→ 대칭 + 멱등 = 정사영 행렬 (9강)")
print("AA⁺ 의 대각합(=랭크) =", round(float(np.trace(A@Ap)), 4))

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[1., 0.], [1., 1.], [1., 2.], [1., 3.]])
b = np.array([1., 3., 4., 6.])     # 한 직선 위에 없는 점 4개

x1 = np.linalg.pinv(A) @ b
x2 = np.linalg.lstsq(A, b, rcond=None)[0]
x3 = np.linalg.solve(A.T @ A, A.T @ b)     # 정규방정식 (10강)

print("pinv      =", x1)
print("lstsq     =", x2)
print("정규방정식 =", x3)
print("셋이 같은가:", np.allclose(x1, x2) and np.allclose(x1, x3))
print("\n잔차 ‖Ax−b‖ =", round(float(np.linalg.norm(A@x1 - b)), 4), " ← 0 이 아니다(해가 없으니)")

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
A = rng.normal(size=(5, 3)) @ rng.normal(size=(3, 4))   # 5×4, 랭크 3
Ap = np.linalg.pinv(A)

조건 = {
    "① A A⁺ A = A":     np.allclose(A @ Ap @ A, A),
    "② A⁺ A A⁺ = A⁺":   np.allclose(Ap @ A @ Ap, Ap),
    "③ (A A⁺)ᵀ = A A⁺": np.allclose((A @ Ap).T, A @ Ap),
    "④ (A⁺ A)ᵀ = A⁺ A": np.allclose((Ap @ A).T, Ap @ A),
}
for k, v in 조건.items():
    print(k, "→", v)
print("\nA 의 랭크 =", np.linalg.matrix_rank(A), " (정사각도 아니고 꽉 차지도 않음)")

In [ ]:
import numpy as np
np.set_printoptions(precision=2, suppress=True)

영화 = ["액션1", "액션2", "로맨스1", "로맨스2"]
사람 = list("ABCDE")
R = np.array([[5., 4., 1., 0.],    # 0 = 아직 안 봄
              [4., 5., 1., 1.],
              [1., 1., 5., 4.],
              [1., 0., 4., 5.],
              [5., 5., 0., 1.]])

U, s, Vt = np.linalg.svd(R, full_matrices=False)
print("특이값 =", s)
print("→ 2개 다음에 뚝 떨어진다. 취향 축은 사실상 2개")

k = 2
R2 = U[:, :k] @ np.diag(s[:k]) @ Vt[:k]
print("\n랭크 2 복원:")
print(R2)
print("\n빈칸 예측:")
for i, j in [(0, 3), (3, 1), (4, 2)]:
    print(f"  {사람[i]} 의 '{영화[j]}' → {R2[i, j]:.2f} 점")

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

R = np.array([[5., 4., 1., 0.], [4., 5., 1., 1.], [1., 1., 5., 4.],
              [1., 0., 4., 5.], [5., 5., 0., 1.]])
U, s, Vt = np.linalg.svd(R, full_matrices=False)

사람좌표 = pd.DataFrame(np.round(U[:, :2] * s[:2], 2),
                        index=list("ABCDE"), columns=["축1", "축2"])
영화좌표 = pd.DataFrame(np.round(Vt[:2].T, 2),
                        index=["액션1", "액션2", "로맨스1", "로맨스2"],
                        columns=["축1", "축2"])
print("사람의 취향 좌표"); print(사람좌표)
print("\n영화의 성격 좌표"); print(영화좌표)
print("\n두 좌표의 내적이 예측 별점 — A × 액션1 =",
      round(float(사람좌표.loc["A"] @ 영화좌표.loc["액션1"]), 2))

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

A = np.array([[2., 1.],
              [0., 3.]])

U, s, Vt = np.linalg.svd(A)
Q = U @ Vt                       # 회전(또는 반사)
S = Vt.T @ np.diag(s) @ Vt       # 대칭 · 양의 정부호

print("Q =\n", Q)
print("S =\n", S)
print()
print("QS = A 인가:", np.allclose(Q @ S, A))
print("Q 가 직교인가:", np.allclose(Q.T @ Q, np.eye(2)))
print("det Q =", round(float(np.linalg.det(Q)), 4), " → +1 이면 순수 회전")
print("S 가 대칭인가:", np.allclose(S, S.T))
print("S 의 고윳값 =", np.linalg.eigvalsh(S), " → 전부 양수 = 양의 정부호")
print("S 의 고윳값 = A 의 특이값:", np.allclose(np.sort(np.linalg.eigvalsh(S)), np.sort(s)))

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
U0, _ = np.linalg.qr(rng.normal(size=(6, 6)))
V0, _ = np.linalg.qr(rng.normal(size=(6, 6)))
s = np.array([10., 5., 2., 0.5, 1e-6, 1e-9])      # 뒤 두 개가 거의 0
A = U0 @ np.diag(s) @ V0.T

b = A @ np.ones(6)
b_noisy = b + rng.normal(scale=1e-4, size=6)       # 아주 작은 잡음

x_inv  = np.linalg.inv(A) @ b_noisy                # 전부 뒤집는다
x_pinv = np.linalg.pinv(A, rcond=1e-3) @ b_noisy   # 작은 σ 는 버린다

print("정답 x = 전부 1")
print("inv  로 푼 x 의 최대 크기 =", f"{np.abs(x_inv).max():.3e}")
print("pinv 로 푼 x           =", np.round(x_pinv, 3))
print()
print("→ 작은 σ 를 뒤집으면 1e-4 짜리 잡음이", f"{np.abs(x_inv).max():.0e}", "배로 폭발한다.")
print("   버리는 편이 훨씬 정확하다. 이것이 A⁺ 의 rcond 이다.")

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# 문제 1 — 손 풀이: x = t(2,4) 로 두면 2(2t)+4(4t)=20t=10 → t=0.5
A1 = np.array([[2., 4.]])
print("문제 1: 손으로 t=1/2 →", 0.5 * np.array([2., 4.]))
print("        pinv         →", np.linalg.pinv(A1) @ np.array([10.]))

# 문제 2 — 영행렬의 의사역행렬은 영행렬 (모든 σ 가 0 → 전부 0 으로 둔다)
O = np.zeros((3, 2))
print("\n문제 2: O⁺ =\n", np.linalg.pinv(O), " (모양은 2×3 으로 뒤집힘)")
print("        네 조건 만족:",
      all([np.allclose(O@np.linalg.pinv(O)@O, O),
           np.allclose(np.linalg.pinv(O)@O@np.linalg.pinv(O), np.linalg.pinv(O))]))

# 문제 3 — k=1 이면 '취향 축 하나' 뿐, k=4 면 원본을 그대로 복원(빈칸=0)
R = np.array([[5., 4., 1., 0.], [4., 5., 1., 1.], [1., 1., 5., 4.],
              [1., 0., 4., 5.], [5., 5., 0., 1.]])
U, s, Vt = np.linalg.svd(R, full_matrices=False)
for k in (1, 2, 4):
    Rk = U[:, :k] @ np.diag(s[:k]) @ Vt[:k]
    print(f"\n문제 3: k={k}  오차 {np.linalg.norm(R-Rk)/np.linalg.norm(R):.1%}"
          f"  A의 '로맨스2' 예측 {Rk[0,3]:.2f}")
print("\n→ k=1: 축이 하나뿐이라 '액션파/로맨스파' 구분만 남고 세부가 사라진다.")
print("→ k=4: 랭크가 꽉 차 원본을 그대로 복원 → 빈칸이 0 인 채로 돌아온다.")
print("   즉 예측을 만드는 것은 '랭크를 낮춘다'는 제약 그 자체다.")

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)